# DL Masterclass 04: Advanced Optimizers & Loss Landscapes

Welcome to Notebook 04. You have calculated the gradient (the direction of steepest descent). You multiply it by your Learning Rate, and you subtract it from your weights.

This is pure Stochastic Gradient Descent (SGD). And in a billion-dimensional, highly non-convex loss landscape, pure SGD is a disaster. It gets trapped in saddle points, bounces violently across ravines, and converges at a glacial pace.

---

## 🎓 Socratic Deep Check
Ponder this question before we begin. The answer led to the creation of AdamW (the default optimizer for almost all modern AI):

> *"In standard SGD, adding L2 Regularization (Weight Decay) to the loss function is mathematically identical to subtracting a fraction of the weight at every step. In Adam, doing this is mathematically broken and degrades performance. Why did the research community have to invent AdamW to decouple Weight Decay from the gradient update?"*

---

## Table of Contents
1. **The Physics of SGD vs Momentum:** How to roll uphill.
2. **Adaptive Learning Rates (RMSProp/AdaGrad):** Solving the ravine problem.
3. **Adam (Adaptive Moment Estimation):** Combining Velocity and Friction.
4. **The AdamW Correction:** Decoupling Weight Decay.


### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** SGD with momentum is like a ball rolling downhill — it builds speed in consistent directions and dampens oscillations. Adam combines momentum with per-parameter adaptive learning rates, making it the default optimizer for most deep learning. But Adam can generalize WORSE than SGD with proper tuning — there's no free lunch.

**Mental Model:** Vanilla SGD is a hiker walking downhill in fog, only feeling the local slope. SGD + Momentum is the hiker on a skateboard — they build speed in consistent downhill directions and roll through small bumps (local minima). Adam is the hiker with a GPS that adjusts step size per-coordinate — taking big steps in flat, uncharted dimensions and small steps in steep, well-explored ones.

**Common Junior Pitfall:** Using a single fixed learning rate for the entire training run. Seniors use learning rate schedulers: high LR at the start for fast exploration, decaying LR later for fine convergence. The warmup-cosine schedule (used in all modern LLM training) is the current best practice.

---


## 1. The Physics of SGD vs SGDM (Momentum)

Imagine dropping a ball into a bowl with a small dent (local minimum) halfway down the slope. 
*   **Pure SGD** acts like a ball with maximum friction and zero mass. It moves exactly where the slope tells it to move. When it hits the dent, the slope becomes 0. It stops forever. It failed to find the true bottom of the bowl.
*   **Momentum** adds physical mass/velocity to the equation.

$v_{t} = \beta v_{t-1} + (1 - \beta) \nabla L(W)$
$W_{t} = W_{t-1} - \alpha v_{t}$

*Note: $\beta$ is usually 0.9. $\alpha$ is the learning rate.*

Because the gradient calculation is now mixed with the *history* of previous gradients ($v_{t-1}$), if the ball hits a local minimum where the slope is 0, the built-up velocity $v_{t-1}$ physically drags the mathematical update *uphill* and out of the dent, allowing it to continue rolling toward the global minimum.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
sns.set_theme(style="whitegrid")

# -----------------------------------------------------
# IMPLEMENTATION: Visualizing the Ravine Problem
# -----------------------------------------------------

# Let's define a highly skewed 2D loss function (a Ravine)
# High gradient in X direction, very low gradient in Y direction.
def loss_function(x, y):
    return 10 * x**2 + 0.1 * y**2

def gradient(x, y):
    return np.array([20 * x, 0.2 * y])

# Simulate Pure SGD
lr = 0.08
pos_sgd = np.array([-8.0, 9.0])
path_sgd = [pos_sgd.copy()]

for _ in range(50):
    grad = gradient(*pos_sgd)
    pos_sgd = pos_sgd - lr * grad
    path_sgd.append(pos_sgd.copy())

# Simulate Momentum
pos_mom = np.array([-8.0, 9.0])
velocity = np.array([0.0, 0.0])
beta = 0.9
lr_mom = 0.015
path_mom = [pos_mom.copy()]

for _ in range(50):
    grad = gradient(*pos_mom)
    velocity = beta * velocity + (1 - beta) * grad
    pos_mom = pos_mom - lr_mom * velocity
    path_mom.append(pos_mom.copy())

path_sgd = np.array(path_sgd)
path_mom = np.array(path_mom)

plt.figure(figsize=(10, 6))
x_mesh = np.linspace(-10, 10, 100)
y_mesh = np.linspace(-10, 10, 100)
X_grid, Y_grid = np.meshgrid(x_mesh, y_mesh)
Z_grid = loss_function(X_grid, Y_grid)

plt.contour(X_grid, Y_grid, Z_grid, levels=np.logspace(-1, 3, 20), cmap='viridis', alpha=0.5)
plt.plot(path_sgd[:, 0], path_sgd[:, 1], 'ro-', label='Pure SGD (Violent Bouncing)', markersize=4)
plt.plot(path_mom[:, 0], path_mom[:, 1], 'bo-', label='Momentum (Smoothed Path)', markersize=4)
plt.plot(0, 0, 'r*', markersize=15, label="Global Minimum")

plt.title("The Ravine Problem: SGD vs Momentum")
plt.legend()
plt.show()

## 2. Adaptive Learning Rates (RMSProp)

Momentum helps, but notice how we still had to carefully tune the Learning Rate. If it's too high in the X-direction, it bounces. If it's too low in the Y-direction, it takes forever.

**RMSProp** (developed by Geoffrey Hinton) solves this by explicitly tracking the *squared* gradients.

$S_t = \beta S_{t-1} + (1 - \beta) (\nabla L)^2$
$W_t = W_{t-1} - \frac{\alpha}{\sqrt{S_t + \epsilon}} \nabla L$

**The Magic:** If a certain parameter (like our X-axis above) has massive, explosive gradients, the denominator $\sqrt{S_t}$ becomes huge, aggressively dividing and shrinking the effective learning rate just for that parameter. If a parameter has tiny gradients (Y-axis), the denominator is tiny, actively *boosting* its learning rate.

It physically flattens the geometry of the loss landscape.

## 3. Adam (Adaptive Moment Estimation)

Adam simply merges Momentum (Velocity) and RMSProp (Friction) together into one algorithm.

1.  **First Moment (Mean of gradients)**: $m_t = \beta_1 m_{t-1} + (1 - \beta_1) \nabla L$ 
2.  **Second Moment (Variance of gradients)**: $v_t = \beta_2 v_{t-1} + (1 - \beta_2) (\nabla L)^2$ 

Apply Bias Correction (because $m$ and $v$ are initialized at 0, they are skewed early on):
$\hat{m}_t = \frac{m_t}{1 - \beta_1^t}$
$\hat{v}_t = \frac{v_t}{1 - \beta_2^t}$

**Update Rule:** $W_t = W_{t-1} - \alpha \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon}$

Adam is extremely robust out-of-the-box, which is why everyone uses it.

## 4. The AdamW Correction (Decoupling Weight Decay)

### 🎓 DEEP QUESTION ANSWERED
**Q:** *Why was AdamW invented? What was mathematically broken in standard Adam?*

**A:** 
Standard L2 Regularization adds a penalty to the loss function based on the size of the weights: $L_{new} = L_{old} + \lambda W^2$. 

In **SGD**, taking the gradient of $L_{new}$ gives $\nabla L_{old} + 2\lambda W$. When you do the weight update $W_{old} - \alpha (\nabla L + 
2\lambda W)$, it perfectly subtracts a little piece of $W$ away. This is called **Weight Decay**, and it prevents weights from growing too large.

But look at the denominator of **Adam**: $W_t = W_{t-1} - \alpha \frac{\nabla L_{old} + 2\lambda W}{\sqrt{v_t}}$.

If a specific weight $W$ is very large, the $2\lambda W$ penalty makes the gradient massive. But Adam explicitly takes massive gradients, squares them into $v_t$, and *divides by them* to shrink the learning rate! 

**The destructive interaction:** Adam assumes the large L2 regularization penalty is just a 'steep ravine' in the data and actively divides it out to "normalize" it. 

Ilya Loshchilov and Frank Hutter (2017) realized this flaw. They explicitly removed the $2\lambda W$ term from the true gradient calculation, entirely bypassed Adam's complex moving averages and division logic, and manually subtracted the weight decay at the very end of the Adam step.

**AdamW Update:** $W_t = W_t - \alpha \lambda W_{t-1}$.

This simple "decoupling" dramatically improved the generalization of Transformer models, making AdamW the undisputed king of Modern Deep Learning.

---
## ✅ Knowledge Check

### Q1: Why does Adam sometimes generalize worse than SGD?

<details><summary>👉 Answer</summary>

Adam's adaptive per-parameter learning rates can converge to SHARP minima in the loss landscape — these are narrow valleys that perform well on training data but poorly on test data. SGD with momentum tends to find FLAT minima (wide valleys) that generalize better because small perturbations in weights don't drastically change the loss. This is why top Kaggle competitors and LLM training often prefer SGD with momentum + cosine annealing for final training.
</details>

### Q2: Learning rate warmup

<details><summary>👉 Answer</summary>

At the start of training, the model's loss landscape is chaotic — large, random gradients cause the optimizer to take wild steps. Warmup starts with a tiny learning rate and gradually increases it over the first 5-10% of training steps, allowing the model to 'orient' itself before taking large steps. Without warmup, Adam can diverge in the first 100 steps of training, especially with large batch sizes.
</details>

### Q3: Batch size and learning rate relationship

<details><summary>👉 Answer</summary>

When you double the batch size, you should increase the learning rate by sqrt(2) (linear scaling rule). Larger batches produce more stable gradient estimates (less noise), so you can afford larger steps. But too-large batches with too-high LR converge to sharp minima. The sweet spot for most tasks is batch_size=32-256 with LR=1e-3 to 1e-4 for Adam.
</details>


---
## 🔨 Practical Practice

### Exercise 1: Implement SGD, SGD+Momentum, and Adam from scratch in NumPy. Train a 3-layer MLP on make_moons. Plot loss curves for all three and observe that Adam converges fastest but SGD+Momentum can find a better final loss.

### Exercise 2: Implement the cosine annealing learning rate schedule from scratch. Plot the LR over training steps. Apply it to Adam and compare convergence vs constant LR training. Show cosine annealing achieves lower final test loss.

### Exercise 3: Visualize the loss landscape: train a simple 2-parameter model, compute loss on a 2D grid of parameter values, and plot the 3D surface. Overlay the optimization trajectory for SGD, Momentum, and Adam to show how each navigates the landscape differently.
